# Merge EPA Water Quality Measurements with Climate Data

In [ ]:
import pandas as pd
import os

# Load EPA merged data
epa_df = pd.read_csv('../../data/tabular/merged/epa-merged.csv')

# Load Climate data
climate_df = pd.read_csv('../../data/tabular/climate/clean/isu-climate-clean.csv')

# Load Mapping data
mapping_df = pd.read_csv('../../data/tabular/merged/epa-to-climate-station-map.csv')

In [ ]:
# 1. Join EPA data with the mapping to get the climate station for each EPA location
mapping_subset = mapping_df[['MonitoringLocationIdentifier', 'climate_station', 'climate_station_name', 'distance_to_climate_station_km']]
epa_mapped = epa_df.merge(mapping_subset, on='MonitoringLocationIdentifier', how='left')

# 2. Extract date from ActivityStartDateTime to merge with climate 'day'
epa_mapped['day'] = pd.to_datetime(epa_mapped['ActivityStartDateTime']).dt.strftime('%Y-%m-%d')

# 3. Merge with climate data on (climate_station, day)
merged_df = epa_mapped.merge(
    climate_df, 
    left_on=['climate_station', 'day'], 
    right_on=['station', 'day'], 
    how='left'
)

# Clean up redundant columns
if 'station' in merged_df.columns:
    merged_df = merged_df.drop(columns=['station'])

In [ ]:
print(f"Merged shape: {merged_df.shape}")
merged_df.head()

In [ ]:
# Save the result
output_path = '../../data/tabular/merged/epa-climate-merged.csv'
merged_df.to_csv(output_path, index=False)
print(f"Saved merged data to {output_path}")